In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
current_dir = Path.cwd()
project_root = current_dir.parents[1]

# Input Path (Reading from the Staging folder)
staging_data_path = project_root / 'data' / 'staging' / 'PPH' / 'stg_pph.csv'

# Output Path (Writing to the Marts folder for Power BI)
marts_output_path = project_root / 'data' / 'marts' 

# Ensure the Marts directory exists
marts_output_path.mkdir(parents=True, exist_ok=True)

print(f"Project Root detected: {project_root}")
print(f"Reading Staging Data from: {staging_data_path}")

In [ ]:
df_staging = pd.read_csv(staging_data_path)

In [ ]:
print(df_staging.head())

In [ ]:
# These columns identify the row (We want to keep these)
id_cols = ['Office Of Earlier Examination', 'Office of Later Examination', 'Year']

# These columns contain the values we want to stack (National & PCT)
value_cols = ['National', 'PCT']

# The melt function converts "Wide" data to "Tall" data
df_transformed = df_staging.melt(
    id_vars=id_cols, 
    value_vars=value_cols, 
    var_name='Filing Route',       # The new column that will say "National" or "PCT"
    value_name='Application Count' # The new column that will hold the numbers
)

# Convert the count to integer (removes decimals like 20.0 -> 20)
df_transformed['Application Count'] = df_transformed['Application Count'].fillna(0).astype(int)

In [ ]:
# Rename columns to standard, business-friendly names for Power BI
df_marts = df_transformed.rename(columns={
    'Office Of Earlier Examination': 'Earlier Office',
    'Office of Later Examination': 'Later Office',
    'Year': 'year',
    'Filing Route': 'Filing Route',
    'Application Count': 'Count'
})

# Save the final file
output_file = marts_output_path / 'fact_pph.csv'
df_marts.to_csv(output_file, index=False)